# Phase 6 — Combined Graph + ML Export
**Marina Osmolovska — MaCAD S3 Graph ML Final**

Builds the access graph for the **combined section**: intact rooms + damaged block + memorial intervention.

**Room type assignments in this graph:**
- Intact section rooms — real `room_type` (bedroom, kitchen, …)
- ALL surviving rooms in damaged block — `corridor` (neutral placeholder; ML predicts their new function)
- Memorial slabs (replacing destroyed rooms) — `balcony` (outdoor/semi-outdoor, zoning 3)
- Step-bridges — `stairs`

Ends with ML preparation using the helpers from S06-15B.

---
**Before running the ML cells at the bottom:** open `S06-15B GML Prepare Graph for Node Classification.ipynb` and run all its cells in **this same kernel**. That defines `CheckMSDGraphPreparation` and `EncodeMSDGraphFeatures`.

## 1. Import the needed classes

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
import os

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer

In [ ]:
renderer = "vscode"

## 4. Set paths to combined OBJ files

After the Rhino Phase 5 design, export the combined section as new OBJ files.
Update the filenames here to match what you exported.

**Naming convention used here:**
- `rooms_combined.obj` — all rooms: intact (real names) + damaged surviving (named `corridor`) + slabs (named `balcony`) + bridges (named `stairs`)
- `doors_entrance_combined.obj` — entrance apertures (same as Phase 3 + any new ones)
- `doors_door_combined.obj` — door apertures (same + new intervention connections)
- `doors_passage_combined.obj` — passage apertures (same + new)

In [ ]:
BASE       = os.path.dirname(os.path.abspath("__file__"))  # Final/ directory
OBJ_DIR    = os.path.join(BASE, "assets", "3d")
ROOMS_OBJ  = os.path.join(OBJ_DIR, "rooms_combined.obj")
DOORS_ENT  = os.path.join(OBJ_DIR, "doors_entrance_combined.obj")
DOORS_DOOR = os.path.join(OBJ_DIR, "doors_door_combined.obj")
DOORS_PASS = os.path.join(OBJ_DIR, "doors_passage_combined.obj")
DATASET_PATH = os.path.join(BASE, "dataset")
os.makedirs(DATASET_PATH, exist_ok=True)

print("OBJ_DIR:", OBJ_DIR)
for p in [ROOMS_OBJ, DOORS_ENT, DOORS_DOOR, DOORS_PASS]:
    print(os.path.basename(p), "exists:", os.path.exists(p))
print("DATASET_PATH:", DATASET_PATH)

## 5. Load combined rooms OBJ

In [ ]:
objects = Topology.ByOBJPath(ROOMS_OBJ)
print("Objects is a list")
print(objects)

## 6. Prepare room dictionaries

Same logic as Phase 3. The room names in the combined OBJ already encode the assignment:
- Intact rooms: their real name (bedroom, kitchen, …)
- All damaged-block surviving rooms: `corridor`
- Memorial slabs: `balcony`
- Step-bridges: `stairs`

In [ ]:
ROOM_LABEL = {
    "bedroom": 0, "livingroom": 1, "kitchen": 2, "dining": 3,
    "corridor": 4, "stairs": 5, "storeroom": 6, "bathroom": 7, "balcony": 8
}
ROOM_COLOR = {
    "bedroom": "blue", "livingroom": "yellow", "kitchen": "orange",
    "dining": "orange", "corridor": "yellow", "stairs": "red",
    "storeroom": "purple", "bathroom": "purple", "balcony": "green"
}

cells_list = []
selectors  = []

for obj in objects:
    d = Topology.Dictionary(obj)
    room_type = (Dictionary.ValueAtKey(d, "name") or "").strip().lower()
    if not room_type:
        room_type = (Dictionary.ValueAtKey(d, "group") or "").strip().lower()
    if room_type not in ROOM_LABEL:
        print(f"WARNING: '{room_type}' not recognised — skipping")
        continue
    faces = Topology.Faces(obj)
    c = Cell.ByFaces(faces) if len(faces) > 1 else faces[0]
    c = Topology.RemoveCollinearEdges(c)
    color = ROOM_COLOR.get(room_type, "grey")
    label = ROOM_LABEL[room_type]
    d = Dictionary.SetValuesAtKeys(d, ["room_type", "label", "color", "vertex_size"],
                                      [room_type,   label,  color,  20])
    s = Topology.InternalVertex(c)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)
    cells_list.append(c)
    print(Dictionary.Keys(d), Dictionary.Values(d))

print(len(cells_list), "cells loaded")

## 7. Build CellComplex

In [ ]:
house = CellComplex.ByCells(cells_list)
house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells=True)
house_cells = Topology.Cells(house)
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 8. Show geometry

In [ ]:
Topology.Show(house_cells, faceColorKey="color", faceOpacity=0.7, renderer=renderer)

## 9. Adjacency graph (g1)

In [ ]:
g1 = Graph.ByTopology(house)
verts = Graph.Vertices(g1)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 10. Show geometry and adjacency graph

In [ ]:
Topology.Show(g1, house, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="white", renderer=renderer)

## 11. Load combined door apertures

In [ ]:
apertures = []

objs = Topology.ByOBJPath(DOORS_ENT)
for obj in objs:
    face = Topology.Faces(obj)[0]
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["door_type", "color", "vertex_size"], ["entrance_door", "green", 15])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)

objs = Topology.ByOBJPath(DOORS_DOOR)
for obj in objs:
    face = Topology.Faces(obj)[0]
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["door_type", "color", "vertex_size"], ["door", "brown", 15])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)

objs = Topology.ByOBJPath(DOORS_PASS)
for obj in objs:
    face = Topology.Faces(obj)[0]
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["door_type", "color", "vertex_size"], ["passage", "grey", 15])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)

print(apertures)
print(len(apertures), "apertures loaded")

## 12. Add apertures to the CellComplex

In [ ]:
house = Topology.AddApertures(house, apertures, subTopologyType="face")

## 13. Access graph (g2)

In [ ]:
g2 = Graph.ByTopology(
    house,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=False,
    useInternalVertex=True,
    storeBRep=False,
    tolerance=0.0001
)
verts = Graph.Vertices(g2)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 14. Show access graph

In [ ]:
Topology.Show(house, apertures, g2, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="white", renderer=renderer)

## 15. Compare with intact section (Phase 4)

Run the same NetworkX metrics on the combined graph and compare with the Phase 4 table.
The delta — especially betweenness and closeness collapsing for orphaned rooms — is the analytical result.

In [ ]:
import networkx as nx
from topologicpy.Edge import Edge
from topologicpy.Vertex import Vertex

if hasattr(Graph, "NetworkXGraph"):
    nxGraph = Graph.NetworkXGraph(g2)
else:
    nxGraph = nx.Graph()
    vertices = Graph.Vertices(g2)
    for i, v in enumerate(vertices):
        d = Topology.Dictionary(v)
        nxGraph.add_node(i, room_type=Dictionary.ValueAtKey(d, "room_type") or "unknown",
                            color=Dictionary.ValueAtKey(d, "color") or "grey")
    for e in Graph.Edges(g2):
        sv = Edge.StartVertex(e)
        ev = Edge.EndVertex(e)
        si = next((i for i, v in enumerate(vertices) if Vertex.Distance(sv, v) < 0.001), None)
        ei = next((i for i, v in enumerate(vertices) if Vertex.Distance(ev, v) < 0.001), None)
        if si is not None and ei is not None:
            de = Topology.Dictionary(e)
            nxGraph.add_edge(si, ei, door_type=Dictionary.ValueAtKey(de, "door_type") or "door")

# Entrance node
entrance_node = None
for u, v, data in nxGraph.edges(data=True):
    if data.get("door_type") == "entrance_door":
        for n in [u, v]:
            if nxGraph.nodes[n].get("room_type") in ("stairs", "corridor"):
                entrance_node = n
                break
    if entrance_node is not None:
        break
if entrance_node is None:
    for n, data in nxGraph.nodes(data=True):
        if data.get("room_type") in ("stairs", "corridor"):
            entrance_node = n
            break

depth       = nx.shortest_path_length(nxGraph, source=entrance_node)
betweenness = nx.betweenness_centrality(nxGraph)
closeness   = nx.closeness_centrality(nxGraph)
degree_c    = nx.degree_centrality(nxGraph)

print("COMBINED GRAPH METRICS (compare with Phase 4 table)")
print(f"{'node':<6} {'room_type':<14} {'depth':>5} {'betweenness':>12} {'closeness':>10} {'degree':>8}")
print("-" * 62)
for n, data in sorted(nxGraph.nodes(data=True), key=lambda x: depth.get(x[0], 99)):
    print(
        f"{n:<6} {data.get('room_type','?'):<14}"
        f"{depth.get(n,-1):>5}"
        f"{betweenness[n]:>13.4f}"
        f"{closeness[n]:>11.4f}"
        f"{degree_c[n]:>9.4f}"
    )

---
## ML Preparation

**Run all cells of `S06-15B GML Prepare Graph for Node Classification.ipynb` in this kernel first.**
That defines `CheckMSDGraphPreparation` and `EncodeMSDGraphFeatures`.

Then run the three cells below.

In [ ]:
# Requires CheckMSDGraphPreparation from S06-15B
summary = CheckMSDGraphPreparation(g2, roomTypeKey="room_type", doorTypeKey="door_type")
# Fix any WARNING items before proceeding to EncodeMSDGraphFeatures

In [ ]:
# Requires EncodeMSDGraphFeatures from S06-15B
g2 = EncodeMSDGraphFeatures(g2)

In [ ]:
Graph.ExportToCSV(g2, path=DATASET_PATH)
print("Exported to:", DATASET_PATH)

In [ ]:
# S06-15C requires train_mask / val_mask / test_mask columns in nodes.csv.
# If Graph.ExportToCSV omitted them, this cell adds them.
import pandas as pd

nodes_path = os.path.join(DATASET_PATH, "nodes.csv")
nodes_df = pd.read_csv(nodes_path)
print("nodes.csv columns:", list(nodes_df.columns))

if "test_mask" not in nodes_df.columns:
    nodes_df["train_mask"] = False
    nodes_df["val_mask"]   = False
    nodes_df["test_mask"]  = True
    nodes_df.to_csv(nodes_path, index=False)
    print("Added mask columns — done.")
else:
    print("Mask columns already present.")

print(nodes_df.head())